# 56 · What changes at organisation scale
Level L6 · Part 7 · deep · ~35 min · builds on 14, 54, 41 (also uses 38, 55)

## Goal
Size Scout for 5,000 users on paper before launch: tokens per minute at the busiest minute against the
provider's tiers, how many runs are in flight at once (Little's law), what the cache hit rate changes,
how much burst headroom to keep, and the monthly bill. Every number starts from real recorded runs.

## The idea
Five thousand users hit rate limits at the busiest minute, not on average. **Capacity maths** turns a few measured numbers (tokens, calls and seconds per run) and a few business
guesses (active users, questions a day, peak hour) into the numbers a provider sells: tokens per minute,
requests per minute, dollars a month. **Little's law** gives concurrency: runs in flight = arrival rate
x time per run. Keep **headroom** for bursts, and check what caching does to tier and bill.

Analogy: a restaurant planning Friday night. Covers per hour times minutes per table gives the tables
needed; you add a few for the rush, and you check the kitchen's limit, not the average day's.

## Picture
```mermaid
flowchart LR
    M["measured per run<br/>tokens, calls, seconds"] --> T["tokens per minute"]
    D["demand guesses<br/>users, questions, peak hour"] --> P["busiest minute<br/>runs per minute"]
    P --> T
    P --> L["Little's law<br/>runs in flight"]
    C["cache hit rate"] --> T
    C --> B["monthly bill"]
    T --> TI{"tier with<br/>headroom?"}
    B --> TI
```

## Already know this?
This is the first lesson of Part 7. If you can answer all of these, skip to lesson 60 (the capstone).
1. Runs arrive at 2/s and each takes 15 s. How many are in flight on average?
2. On Anthropic, why does an 80% cache hit rate let you send 5x more input per minute?
3. Your provider fails for 20 minutes. What does Scout do?
4. Name one lesson from a published production agent write-up.
5. Name two vendor API changes this year.

Questions 3 to 5 preview the rest of Part 7; if you can't answer them, continue with lesson 56 anyway.

<details><summary>Short answers</summary>
1. 30 (Little's law). 2. Only uncached input counts toward ITPM. 3. Fails over to a
second provider (57). 4. E.g. start with the simplest design (58).
5. E.g. Gemini Interactions API GA, OpenAI Assistants API shut down (59).
</details>

## Step by step
### 1. Start from measured runs
No model calls in this lesson: sizing starts from numbers you already measured. Lesson 55 saved 20 real
Scout runs on `gemini-3.5-flash-lite` (recorded output) and lesson 41 saved a 5-turn chat with a long,
cached prompt (recorded output). The price table is lesson 54's, plus the cached-input price.

In [1]:
import json, math, random, statistics
from pathlib import Path

RUNS = json.loads(Path("../data/recorded/55.json").read_text())["eval"]      # recorded output, lesson 55
CACHED_CHAT = json.loads(Path("../data/recorded/41.json").read_text())["fixed"]  # recorded output, lesson 41
# USD per 1M tokens; input/output from the Gemini pricing page 2026-10-06.
# cached = ESTIMATE: 10% of input, from Vertex's documented 90% cached discount; not a listed price
PRICES_PER_MTOK = {
    "gemini-3.5-flash-lite": {"input": 0.30, "cached": 0.03, "output": 2.50},
    "gemini-3.8-flash":      {"input": 0.75, "cached": 0.075, "output": 3.75},  # promo until 2026-12-31
}
print(len(RUNS), "recorded Scout runs | first run's calls:", RUNS[0]["calls"])

20 recorded Scout runs | first run's calls: [{'in': 157, 'seconds': 0.628, 'out': 16}, {'in': 246, 'seconds': 0.586, 'out': 17}]


### 2. One run, in numbers
An agent run is several model calls, and each call re-sends the conversation so far. So the unit to size
is the **run**: calls, input and output tokens summed over its calls, and model seconds. Look at the
mean and the p95 (lesson 55's percentile).

In [2]:
def percentile(xs: list[float], p: float) -> float:
    xs = sorted(xs)
    return xs[math.ceil(p / 100 * len(xs)) - 1]

def per_run(row: dict) -> dict:
    calls = row["calls"]
    return {"calls": len(calls), "in": sum(c["in"] for c in calls),
            "out": sum(c["out"] for c in calls), "seconds": sum(c["seconds"] for c in calls)}

TOY = [per_run(r) for r in RUNS]
for key in ("calls", "in", "out", "seconds"):
    vals = [t[key] for t in TOY]
    print(f"{key:8} mean {statistics.mean(vals):8.2f}   p95 {percentile(vals, 95):8.2f}")

calls    mean     2.30   p95     3.00
in       mean   477.10   p95   692.00
out      mean    44.15   p95    79.00
seconds  mean     1.37   p95     1.86


About 2.3 calls, ~480 input tokens, ~44 output tokens and ~1.4 s per run. p95 is close to the mean
(1.86 s vs 1.37 s, 3 calls vs 2.3), so the mean is safe here; the burst simulation covers the tail. But these runs used six short
notes and a tiny prompt. A real deployment carries a long system prompt, tool definitions and notes in
every call: lesson 41's chat measured that size, and how much of it the cache served.

In [3]:
prompt_per_call = statistics.mean(r["prompt"] for r in CACHED_CHAT)
hit_rate = sum(r["cached"] for r in CACHED_CHAT) / sum(r["prompt"] for r in CACHED_CHAT)
calls_per_run = statistics.mean(t["calls"] for t in TOY)
RUN = {"calls": calls_per_run, "in": calls_per_run * prompt_per_call,
       "out": statistics.mean(t["out"] for t in TOY), "seconds": statistics.mean(t["seconds"] for t in TOY)}
print(f"prompt per call {prompt_per_call:,.0f} tokens | measured cache hit rate {hit_rate:.0%}")
print("production run:", {k: round(v, 1) for k, v in RUN.items()})

prompt per call 9,883 tokens | measured cache hit rate 82%
production run: {'calls': 2.3, 'in': 22730.0, 'out': 44.1, 'seconds': 1.4}


82% is one measured chat with a stable prefix. It assumes all users share one cached prefix. Treat it
as an upper estimate and re-measure in production.

A production run sends ~22,700 input tokens, not ~480: almost all of it is the fixed prefix, sent again
on every call. Keep the recorded seconds for now; step 7 asks what longer runs do.

### 3. From users to runs per minute
Now the business guesses. Write them down in one place so anyone can challenge them: who is active, how
often they ask, and how much of the day lands in the busiest hour.

In [4]:
DEMAND = {
    "users": 5_000,
    "daily_active": 0.60,        # share of users who use Scout on a workday
    "questions_per_day": 6,      # per active user
    "workdays_per_month": 22,
    "peak_hour_share": 0.15,     # share of a day's runs in the busiest hour
}

def runs_per_day(d: dict) -> float:
    return d["users"] * d["daily_active"] * d["questions_per_day"]

peak_per_min = runs_per_day(DEMAND) * DEMAND["peak_hour_share"] / 60
print(f"{runs_per_day(DEMAND):,.0f} runs a day | average minute of the peak hour: {peak_per_min:.1f} runs")

18,000 runs a day | average minute of the peak hour: 45.0 runs


### 4. The busiest minute: burst headroom
45 runs a minute is an *average* over the peak hour. Users arrive at random, so some minutes get more.
Simulate it: arrivals with random gaps (a Poisson process, seeded), 22 peak hours, and find the busiest
minute. The ratio busiest / average is the **burst factor**.

In [5]:
def arrivals(rate_per_min: float, minutes: int, rng: random.Random) -> list[float]:
    t, out = 0.0, []
    while (t := t + rng.expovariate(rate_per_min)) < minutes:
        out.append(t)                                    # arrival times, in minutes
    return out

rng = random.Random(56)
busiest = []
for day in range(DEMAND["workdays_per_month"]):
    counts = [0] * 60
    for t in arrivals(peak_per_min, 60, rng):
        counts[int(t)] += 1
    busiest.append(max(counts))
burst = max(busiest) / peak_per_min
print(f"busiest minute each day: {sorted(busiest)}")
print(f"worst minute of the month: {max(busiest)} runs = {burst:.2f}x the average")

busiest minute each day: [54, 58, 59, 59, 59, 59, 59, 60, 60, 60, 61, 62, 62, 62, 62, 63, 63, 64, 65, 66, 67, 68]
worst minute of the month: 68 runs = 1.51x the average


Pure chance alone gives a minute 1.5x busier than the average.
Real traffic is lumpier than this (everyone opens Scout after the 9:00 meeting).
So plan for the simulated burst *and* keep spare room on top:
here, never plan to use more than 70% of a tier.

### 5. Tokens per minute against the provider's tiers
Scout runs on Vertex AI. Its Standard PayGo tiers give each Flash-Lite model a baseline in tokens per
minute, per organisation, and the tier you get depends on what you spent in the last 30 days.

In [6]:
VERTEX_FLASH_LITE_TPM = [       # Vertex Standard PayGo, docs 2026-10-09: (tier, 30-day spend, baseline TPM)
    ("Tier 1", "$10-$250", 2_000_000), ("Tier 2", "$250-$2,000", 4_000_000),
    ("Tier 3", "$2,000-$50,000", 10_000_000), ("Tier 4", ">$50,000", 50_000_000),
]
MAX_UTIL = 0.70                 # headroom: plan to use at most 70% of a tier

def smallest_tier(need_tpm: float, tiers=VERTEX_FLASH_LITE_TPM) -> str:
    for name, spend, tpm in tiers:
        if need_tpm <= MAX_UTIL * tpm:
            return f"{name} ({spend} a month): {need_tpm / tpm:.0%} of {tpm / 1e6:.0f}M TPM"
    return "no PayGo tier fits: Provisioned Throughput or a second model"

The page does not say whether cached tokens count toward TPM, so count them all (the safe side). Compare
the toy runs, the average peak minute and the busiest minute.

In [7]:
def tpm(runs_per_min: float, run: dict) -> float:
    return runs_per_min * (run["in"] + run["out"])

toy_run = {k: statistics.mean(t[k] for t in TOY) for k in ("in", "out")}
for label, rpm, run in [("toy prompt, busiest minute", max(busiest), toy_run),
                        ("real prompt, average minute", peak_per_min, RUN),
                        ("real prompt, busiest minute", max(busiest), RUN)]:
    need = tpm(rpm, run)
    print(f"{label:28} {need:>11,.0f} TPM -> {smallest_tier(need)}")

toy prompt, busiest minute        35,445 TPM -> Tier 1 ($10-$250 a month): 2% of 2M TPM
real prompt, average minute    1,024,836 TPM -> Tier 1 ($10-$250 a month): 51% of 2M TPM
real prompt, busiest minute    1,548,641 TPM -> Tier 2 ($250-$2,000 a month): 39% of 4M TPM


The toy prompt would fit anywhere. The real prompt fits Tier 1 at the average minute (51%).
But the busiest minute uses 77% of it, past the 70% plan:
sizing from the average would pick the wrong tier.
Requests per minute are no concern here (about 2.3 calls per run); Vertex PayGo has no RPM limit at all.

### 6. What caching changes: Anthropic's tiers
Anthropic counts limits differently: input tokens per minute (ITPM) and output tokens per minute (OTPM),
and **only uncached input counts toward ITPM**. If Scout ran on Claude, the cache hit rate would change
which tier fits. (Claude's tokenizer counts the same text differently; treat this as a rough check.)

In [8]:
ANTHROPIC_TIERS = [   # per model class, RESEARCH.md section 7: (tier, RPM, ITPM, OTPM)
    ("Start", 1_000, 2_000_000, 400_000), ("Build", 5_000, 5_000_000, 1_000_000),
    ("Scale", 10_000, 10_000_000, 2_000_000),
]

def anthropic_tier(runs_per_min: float, run: dict, hit: float) -> str:
    need = (runs_per_min * run["calls"], runs_per_min * run["in"] * (1 - hit), runs_per_min * run["out"])
    for name, *limits in ANTHROPIC_TIERS:
        if all(n <= MAX_UTIL * lim for n, lim in zip(need, limits)):
            return f"{name}: RPM {need[0]:,.0f}, ITPM {need[1]:,.0f}, OTPM {need[2]:,.0f}"
    return f"none fits: ITPM {need[1]:,.0f}"

for hit in (0.0, hit_rate):
    print(f"cache hit {hit:4.0%} -> {anthropic_tier(max(busiest), RUN, hit)}")

cache hit   0% -> Build: RPM 156, ITPM 1,545,639, OTPM 3,002
cache hit  82% -> Start: RPM 156, ITPM 271,917, OTPM 3,002


With no caching Scout needs the Build tier; at the measured 82% hit rate the smallest tier carries it with
room to spare. A cache hit rate is a capacity number, not only a cost number.

### 7. How many runs are in flight: Little's law
**Little's law**: in a steady system, the average number of items inside = arrival rate x average time
each one stays, `L = λ x W`. Units must match: runs per *second* times *seconds* per run.

In [9]:
lam = max(busiest) / 60                          # runs per second at the busiest minute
W = RUN["seconds"]                               # recorded model seconds per run
print(f"λ = {lam:.2f} runs/s, W = {W:.2f} s  ->  L = {lam * W:.2f} runs in flight on average")

λ = 1.13 runs/s, W = 1.37 s  ->  L = 1.55 runs in flight on average


About one and a half. Check the law instead of trusting it: simulate an hour of arrivals at that rate,
give each run the duration of a random recorded run, and count runs in flight every 0.1 s.

In [10]:
def in_flight(rate_per_s: float, durations: list[float], seconds: int, rng: random.Random) -> list[int]:
    starts = [t * 60 for t in arrivals(rate_per_s * 60, seconds // 60, rng)]
    ends = [s + rng.choice(durations) for s in starts]
    events = sorted([(s, 1) for s in starts] + [(e, -1) for e in ends])
    samples, live, i = [], 0, 0
    for k in range(seconds * 10):                # one sample every 0.1 s
        while i < len(events) and events[i][0] <= k / 10:
            live, i = live + events[i][1], i + 1
        samples.append(live)
    return samples

seen = in_flight(lam, [t["seconds"] for t in TOY], 3_600, random.Random(56))
print(f"simulated mean {statistics.mean(seen):.2f} | p99 {percentile(seen, 99)} | max {max(seen)}")

simulated mean 1.53 | p99 5 | max 8


The simulated mean (1.53) matches `λ x W` (1.55).
But for 1% of the hour 5 or more runs were in flight, and once 8.
Size a concurrency limit (lesson 38's semaphore) and a worker pool with the p99 and the maximum, not the mean.

Now the honest question: these recorded runs took 1.4 s. Longer research tasks take far longer per run.

In [11]:
for W_s in (1.4, 10, 30, 90):
    L = lam * W_s
    seen = in_flight(lam, [W_s], 3_600, random.Random(56))
    print(f"W = {W_s:>4} s -> L = {L:6.1f} mean | simulated p99 {percentile(seen, 99):>3}")

W =  1.4 s -> L =    1.6 mean | simulated p99   5
W =   10 s -> L =   11.3 mean | simulated p99  20
W =   30 s -> L =   34.0 mean | simulated p99  47
W =   90 s -> L =  102.0 mean | simulated p99 127


Concurrency grows in step with run time (90 s runs: ~100 in flight, p99 127).
Tokens per minute do not change (same tokens, spread over a longer run).
Long agent runs are what fill worker pools and open connections.

### 8. The monthly bill, and the tier it earns
Cost per run with a cache hit rate: cached input at the cached price, the rest at the full price, output
at the output price. Then runs per month. Cached prices are an estimate; Google's page does not list them.

In [12]:
def run_cost(run: dict, model: str, hit: float) -> float:
    p = PRICES_PER_MTOK[model]
    return (run["in"] * ((1 - hit) * p["input"] + hit * p["cached"]) + run["out"] * p["output"]) / 1e6

runs_month = runs_per_day(DEMAND) * DEMAND["workdays_per_month"]
for model in PRICES_PER_MTOK:
    for hit in (0.0, hit_rate):
        c = run_cost(RUN, model, hit)
        print(f"{model:22} hit {hit:4.0%}: ${c:.5f}/run -> ${c * runs_month:>9,.0f} a month")

gemini-3.5-flash-lite  hit   0%: $0.00693/run -> $    2,744 a month
gemini-3.5-flash-lite  hit  82%: $0.00187/run -> $      741 a month
gemini-3.8-flash       hit   0%: $0.01721/run -> $    6,816 a month
gemini-3.8-flash       hit  82%: $0.00457/run -> $    1,810 a month


On Vertex the tier follows the last 30 days of spend, so a cheaper Scout earns a *lower* tier. Check that
the tier the bill earns can carry the busiest minute.

In [13]:
def earned_tier(monthly_usd: float) -> tuple[str, int]:
    bounds = [(250, 0), (2_000, 1), (50_000, 2), (math.inf, 3)]
    idx = next(i for limit, i in bounds if monthly_usd <= limit)
    return VERTEX_FLASH_LITE_TPM[idx][0], VERTEX_FLASH_LITE_TPM[idx][2]

need = tpm(max(busiest), RUN)
for hit in (0.0, hit_rate):
    bill = run_cost(RUN, "gemini-3.5-flash-lite", hit) * runs_month
    name, cap = earned_tier(bill)
    print(f"hit {hit:4.0%}: ${bill:,.0f}/month earns {name} ({cap / 1e6:.0f}M TPM) | busiest minute uses {need / cap:.0%}")

hit   0%: $2,744/month earns Tier 3 (10M TPM) | busiest minute uses 15%
hit  82%: $741/month earns Tier 2 (4M TPM) | busiest minute uses 39%


Caching cuts the bill by about 73%, and the cached bill still earns Tier 2, where the busiest minute
uses 39%. But a new account starts in Tier 1, where it would use 77%: launch to a share of the users
first and grow as the spend history builds.

### 9. One function for the whole plan
Put it together so the plan can be re-run when a guess changes. It returns the numbers a launch review
asks for.

In [14]:
def size_scout(demand: dict, run: dict, model: str, hit: float, burst: float) -> dict:
    per_min = runs_per_day(demand) * demand["peak_hour_share"] / 60 * burst
    need = tpm(per_min, run)
    bill = run_cost(run, model, hit) * runs_per_day(demand) * demand["workdays_per_month"]
    earned, cap = earned_tier(bill)
    return {"busiest runs/min": round(per_min), "TPM needed": round(need), "tier needed": smallest_tier(need),
            "in flight (mean)": round(per_min / 60 * run["seconds"], 1), "monthly $": round(bill),
            "bill earns": f"{earned}, busiest minute uses {need / cap:.0%}"}

for users in (5_000, 50_000):
    plan = size_scout(DEMAND | {"users": users}, RUN, "gemini-3.5-flash-lite", hit_rate, burst)
    print(f"{users:,} users:", json.dumps(plan, indent=1))
print(f"50,000 users, average peak minute: {tpm(peak_per_min * 10, RUN):,.0f} TPM")

5,000 users: {
 "busiest runs/min": 68,
 "TPM needed": 1548641,
 "tier needed": "Tier 2 ($250-$2,000 a month): 39% of 4M TPM",
 "in flight (mean)": 1.5,
 "monthly $": 741,
 "bill earns": "Tier 2, busiest minute uses 39%"
}
50,000 users: {
 "busiest runs/min": 680,
 "TPM needed": 15486408,
 "tier needed": "Tier 4 (>$50,000 a month): 31% of 50M TPM",
 "in flight (mean)": 15.5,
 "monthly $": 7413,
 "bill earns": "Tier 3, busiest minute uses 155%"
}
50,000 users, average peak minute: 10,248,358 TPM


At 50,000 users the busiest minute needs ~15.5M TPM, which only Tier 4 holds with headroom. But the
cached bill (~$7,400) earns Tier 3, where that minute is 155% of the baseline. Caching made Scout too
cheap to earn the tier it needs: buy Provisioned Throughput, or split traffic across two models or
providers (lesson 57).

The burst factor (1.51x) was simulated at 45 runs a minute. At 450 a minute it would be smaller,
so 155% is conservative. Even the average minute (~10.2M TPM) exceeds Tier 3's 10M.

## At real scale
- **Limits are per organisation, earned by spend.** Vertex PayGo: baseline TPM per model by 30-day spend
  (Flash-Lite 2M to 50M), no RPM limit, best effort above it; a 429 means "temporary high contention".
- **Bursts fail below the average.** Google: "High and instantaneous traffic can lead to throttling even
  if your average per-minute usage is below your limit". Anthropic adds "acceleration limits" for
  sudden ramps. Smooth with a shared limiter (lesson 38).
- **Caching is capacity on Anthropic:** cache reads skip ITPM, so 2M ITPM at 80% hits carries 10M input
  tokens a minute. Vertex does not document this.
- **What breaks first:** the shared organisation. One team's batch job throttles another's chat.
  Fixes: separate workspaces or projects per team, per-team budgets (lesson 54), a second provider (lesson 57).
- **Watch:** busiest-minute TPM / baseline, 429 rate, p99 in flight, hit rate, 30-day spend.

## Decision guide
| Situation | Approach | Why |
|---|---|---|
| Busiest minute < 70% of your tier | Stay on PayGo, client-side limiter | Cheapest; the limiter absorbs bursts |
| Busiest minute over the baseline, short spikes | Queue and smooth, raise headroom | Above baseline is best effort only |
| Need a strict latency SLA | Provisioned Throughput | Assured capacity, paid whether used or not |
| Long prefix sent on every call | Prompt caching, stable prefix (lesson 41) | Cuts cost; on Anthropic also cuts ITPM |
| Big launch on a new account | Ramp users over weeks | Tiers follow 30-day spend |
| Work that can wait hours | Batch API (lesson 42) | 50% off, separate quota |
| One provider is the single point of failure | Second provider via the adapter (lesson 57) | Outages and contention |

## What just happened
- Twenty recorded runs gave 2.3 calls and ~1.4 s per run; lesson 41's recorded prompt made a real run
  ~22,700 input tokens, 82% of them cached.
- 5,000 users became 18,000 runs a day and 45 runs in the average peak minute; a seeded simulation
  found a busiest minute of 68 (1.51x).
- The busiest minute used 77% of Tier 1, past the 70% plan; the average minute (51%) would have fitted.
- On Anthropic's tiers the cache hit rate alone moved Scout from Build to Start.
- Little's law gave 1.55 runs in flight, the simulation 1.53; longer runs raised it in step.
- Caching cut the bill from $2,744 to $741 a month; at 50,000 users the bill earned a tier too small.

## Common mistakes
**1. Sizing from the daily average.** Spread 18,000 runs evenly over 24 hours and any tier looks fine.
A launch check that compares the busiest minute with the tier picked from the average fails.

In [15]:
def check_fits(need_tpm: float, tier_tpm: float) -> None:
    if need_tpm > MAX_UTIL * tier_tpm:
        raise ValueError(f"needs {need_tpm:,.0f} TPM, plan allows {MAX_UTIL * tier_tpm:,.0f}")
    print(f"fits: {need_tpm / tier_tpm:.0%} of the tier")

daily_avg_per_min = runs_per_day(DEMAND) / (24 * 60)
print(f"daily average {daily_avg_per_min:.1f} runs/min -> {tpm(daily_avg_per_min, RUN):,.0f} TPM: 'Tier 1 is plenty'")

daily average 12.5 runs/min -> 284,677 TPM: 'Tier 1 is plenty'


The busiest minute against that Tier 1 choice:

In [16]:
check_fits(tpm(max(busiest), RUN), tier_tpm=2_000_000)

ValueError: needs 1,548,641 TPM, plan allows 1,400,000

Fix: size from the busiest minute of the peak hour, and pick the tier with `smallest_tier`.

In [17]:
print(smallest_tier(tpm(max(busiest), RUN)))

Tier 2 ($250-$2,000 a month): 39% of 4M TPM


**2. Mixing units in Little's law.** Runs per *minute* times *seconds* per run is 60x too big (and per
second times minutes is 60x too small). Compare with the simulation, as in step 7.

In [18]:
L_wrong = max(busiest) * RUN["seconds"]          # runs per minute x seconds
L_sim = statistics.mean(in_flight(lam, [RUN["seconds"]], 3_600, random.Random(56)))
assert abs(L_wrong - L_sim) / L_sim < 0.2, f"formula {L_wrong:.1f} vs simulated {L_sim:.2f}"

AssertionError: formula 92.9 vs simulated 1.53

Fix: convert the rate to per second first.

In [19]:
L_right = max(busiest) / 60 * RUN["seconds"]
print(f"formula {L_right:.2f} vs simulated {L_sim:.2f}")

formula 1.55 vs simulated 1.53


## Try it
The board wants Scout for 20,000 users, on `gemini-3.8-flash` after its promo ends on 2026-12-31
(then $1.50 input, $0.15 cached, $7.50 output per 1M tokens). Copy the price table with the new prices
(do not edit `PRICES_PER_MTOK`), then work out: the busiest-minute TPM, the monthly bill at the measured
hit rate, and whether the tier that bill earns carries the busiest minute. Note: Vertex Flash tiers have
the same TPM as Flash-Lite.

In [20]:
# your code here

## Key terms
- **capacity maths** — turning measured per-run numbers and demand guesses into tokens per minute, concurrency and monthly cost before launch.
- **busiest minute** — the highest-traffic minute you expect; the number to compare with a rate limit, not the average.
- **burst factor** — busiest minute divided by the average minute of the peak hour.
- **burst headroom** — spare capacity kept above the busiest expected minute, here by never planning above 70% of a tier.
- **Little's law** — average items in a system = arrival rate x average time in it (`L = λW`), in matching units.
- **rate-limit tier** — a provider's level of limits (TPM, RPM) for an account, usually earned by past spend.
- **baseline throughput** — Vertex AI's tokens per minute per model that a tier is sized for; traffic above it is served best effort.
- **Provisioned Throughput** — capacity bought in advance from Vertex AI for a strict SLA, paid whether used or not.